# Data Coverage

Packages

In [ ]:
# Public packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import math
import os
import re
import tabulate
from IPython.display import display, Markdown

# Custom packages
from filter import FilterDF as fdf
from benchmarks import ParetoAnalysis as pa
from benchmarks import AccuracyCalculation as ac
from integrity_fixes import DataFixer as fix, DataExporter as exporter

Preemptively set new Pandas option, also set matplotlib to close

In [ ]:
pd.options.mode.copy_on_write = True
%matplotlib inline
%config InlineBackend.close_figures=True

Allow reloading of custom Python classes without reseting kernel

In [ ]:
pd.set_option('display.max_rows', 100)
%load_ext autoreload
%autoreload 2

Load formatted data

In [ ]:
%store -r static_data_merged
%store -r sales_data_merged

Read data from parquet files

In [ ]:
# Check if the data is already imported
if 'static_data_merged' not in locals():

    # Retrieve filnames
    static_data_filenames = os.listdir(f"2_palate_data_parquet_cleaned")

    # Initialize a dictionary for the four parquet files of "static reference data"
    static_data = {}
    for filename in tqdm(static_data_filenames):

        # Exclude other files and folders
        if ".parquet" in filename:
            df = pd.read_parquet(f"2_palate_data_parquet_cleaned/{filename}")
            base_name = re.sub(r'\.parquet$', '', filename)
            static_data[base_name] = df

    # Rename and store
    static_data_merged = static_data.copy()
    %store static_data_merged


# Data already exists
else:
    static_data = static_data_merged.copy()

# Check if the data is already imported
if 'sales_data_merged' not in locals():

    # Retrieve filnames
    restaurant_sales_filenames = os.listdir(f"2_palate_data_parquet_cleaned/orders_item_level")

    # Initialize a dictionary for the 30 parquet files of "restaurant sales data"
    sales_and_menu_data = {}
    for filename in tqdm(restaurant_sales_filenames):
        
        # Exclude other files and folders
        if ".parquet" in filename:
            df = pd.read_parquet(f"2_palate_data_parquet_cleaned/orders_item_level/{filename}")
            location_id = re.sub(r'_sales_and_menu\.parquet$', '', filename)
            sales_and_menu_data[location_id] = df
    
    # Rename and store
    sales_data_merged = {}
    for loc_id, df in sales_and_menu_data.items():
        sales_data_merged[loc_id] = df.copy()
    %store sales_data_merged

# Data already exists
else:
    sales_and_menu_data = {}
    for loc_id, df in sales_data_merged.items():
        sales_and_menu_data[loc_id] = df.copy()


# Rename data for ease of use
before_after_details = static_data['before_after_details'].copy() # Promotional items (30 rows)
customers = static_data['customers'].copy() # Specific customer information: for matching customers with orders
items_tagged = static_data['items_tagged'].copy() # Menu items for all restaurants: for matching plant-based labels with orders
locations = static_data['locations'].copy() # Restaurant details (30 rows)
location_ids = list(sales_and_menu_data.keys())

### Calculating Weekly Data Coverage

In [ ]:
# Calculate active weeks
active_weeks_dict = {}
for loc_id, df in sales_and_menu_data.items():
    active_weeks_dict[loc_id] = (df
                                 .resample('W')
                                 .size()
                                 .to_frame(name='W')
                                 .query('0 < W')
                                 .index
                                 .tz_localize(None)
                                 .to_period('W')
                                 .tolist())

### Weekly Data Coverage Visual

In [ ]:
# Visualizing with gaps for inactive weeks
coverage_fig, ax = plt.subplots(figsize=(14, 8))

# Loop through every active week within a single restaurant
for loc_id, active_weeks in active_weeks_dict.items():

    # Index into the promotional items for this restaurant
    promo_datetime = before_after_details.loc[loc_id,'cross_over_date'].tz_localize('UTC')

    # For every active week
    for week in active_weeks:

        # Place a blue dot
        ax.hlines(y=loc_id, xmin=week.start_time, xmax=week.end_time, colors='blue', lw=2, label=loc_id)

    # Place a red circle for the promotional item
    ax.plot(promo_datetime, loc_id, 'ro', alpha=0.5)

# Plot
ax.set_title('Weekly Activity for Each Restaurant with Gaps for Inactive Weeks')
ax.set_xlabel('Date')
ax.set_ylabel('Restaurant ID')

# Figure
coverage_fig.tight_layout()

# Save
plt.savefig('Data Coverage.png', bbox_inches='tight')
plt.show()

### Data Density: Entire Set, Before Promo, and After Promo

In [ ]:
def coverage_calculator(loc_id, df, freqs=['12H'], periods=['all','b','b2m','a','a2m']):

    # Identify necessary dates
    first_date = df.index[0]
    last_date = df.index[-1]
    promo_datetime = before_after_details.loc[loc_id, 'cross_over_date'].tz_localize('UTC') # Identify introductin date
    two_months_before = promo_datetime - pd.DateOffset(months=2) # Two months before the promotional introduction date
    two_months_after = promo_datetime + pd.DateOffset(months=2) # Two months after the promotional introduction date

    # All time period options
    all_periods = {'all':(first_date, last_date),
                   'b':(first_date, promo_datetime),
                   'b2m':(two_months_before, promo_datetime),
                   'a':(promo_datetime, last_date),
                   'a2m':(promo_datetime, two_months_after)}

    # Filter to chosen time periods for iterating
    periods_to_use = {}
    for period in periods:
        periods_to_use[period] = all_periods[period]

    # Initialize container to aggregate for summary
    row = {'loc_id': loc_id}
    for qualifier, (beginning, end) in periods_to_use.items():
        
        # Unpack beginning and end dates to determine the actual data within the period
        period = df[beginning:end]

        # Calculate total
        row[f'{qualifier}_obs'] = period.size

        # Resample the time series for every frequency
        for freq in freqs:

            # Possible periods
            possible = pd.date_range(beginning, end, freq=freq)

            # Number of active weeks within the bounds, and then before and after
            resampled_data = period.resample(freq).size()
            active_total_at_freq = (0 < resampled_data).sum()
            possible_total = possible.size

            # Calculate data coverage (when the restaurant is active) as a fraction of the total possible days
            coverage_ratio = active_total_at_freq / possible_total
            rounded_coverage_ratio = round(100*coverage_ratio)/100

            # Other simple stats
            rounded_mean = round(np.mean(resampled_data))
            rounded_sd = round(np.std(resampled_data))

            # Store
            # row[f'{qualifier}_{freq}'] = active_total_at_freq
            row[f'{qualifier}_{freq}_coverage'] = rounded_coverage_ratio
            # row[f'{qualifier}_{freq}_mean'] = rounded_mean 
            # row[f'{qualifier}_{freq}_sd'] = rounded_sd

    return row

Apply function

In [ ]:
# Initialize a list to see if there is a data buffer before and after the promotional item introduction
data_coverage_list = []
for loc_id, df in sales_and_menu_data.items():
    row = coverage_calculator(loc_id, df)
    data_coverage_list.append(row)

# Create data frame
data_coverage = pd.DataFrame(data_coverage_list)

View

In [ ]:
data_coverage.sort_values('b2m_12H_coverage', ascending=False)

Visuals

In [ ]:
def plot_time_series(loc_id, df, max_ylim=0, freq='D', subset=True):

    
    # Turn auto display
    plt.ioff()

    # Filter to plant-based items and resample
    plant_based = df.query('is_plant_based == "Yes"')
    promo_datetime = pd.to_datetime(before_after_details.loc[loc_id, 'cross_over_date']).tz_localize('UTC')
    two_months_before = promo_datetime - pd.DateOffset(months=2)
    two_months_after = promo_datetime + pd.DateOffset(months=2)

    # Resample to specified frequency
    plant_based = plant_based.resample(freq)[['item_quantity']].sum()
    all_items = df.resample(freq)[['item_quantity']].sum()

    # Compute ratios and handle missing data
    all_items.replace(0, np.nan, inplace=True)
    plant_based_ratio = plant_based / all_items

    # Subset for the specified time range
    if subset:
        plant_based_ratio = plant_based_ratio[two_months_before:two_months_after]
        all_items = all_items[two_months_before:two_months_after]

    # Identify ends of contiguous data chunks
    is_contiguous = plant_based_ratio.notna()
    shift_plus = is_contiguous.shift(1, fill_value=False)
    shift_minus = is_contiguous.shift(-1, fill_value=False)
    start_points = is_contiguous & ~shift_plus
    end_points = is_contiguous & ~shift_minus

    # Plotting
    fig, ax = plt.subplots(1, 2, figsize=(16, 4))  # Creates a single subplot
    ax1, ax2 = ax

    ## Plot 1

    # Main plot
    ax1.plot(plant_based_ratio.index, plant_based_ratio, marker='o', markersize=1, linewidth=2, label='Plant-Based Items Fraction')

    # Adding dots for the start and end of each contiguous chunk
    ax1.plot(plant_based_ratio[start_points].index, plant_based_ratio[start_points], linewidth=0, color='#2fb7bf', markersize=5, marker='o', label='Start Extant Data')
    ax1.plot(plant_based_ratio[end_points].index, plant_based_ratio[end_points], linewidth=0, color='#1f77c4', markersize=5, marker='o', label='End Extant Data')

    # Promo date line
    ax1.axvline(x=promo_datetime, color='red', linestyle='--', label='Promo Date')

    # Ticks and limits
    xticks = pd.date_range(promo_datetime - pd.DateOffset(days=60), periods=10, freq="15D")
    if subset:
        ax1.set_xticks(ticks=xticks)
        ax1.set_xticklabels(labels=xticks.date, rotation=70)
        ax1.set_xlim(promo_datetime - pd.DateOffset(days=60), promo_datetime + pd.DateOffset(days=60))
    ax1.set_ylim(0, 1)

    # Axis and title
    ax1.set_title(f'Plant-Based Items Fraction for {loc_id}')
    ax1.set_ylabel('Ratio')
    ax1.set_xlabel('Date')
    ax1.legend()

    ## Plot 2

    # Main plot
    ax2.plot(all_items.index, all_items, linewidth=2, color='orange', marker='o', markersize=1, label='Total Item Quantity')

    # Extra details, adding dots to the edge of non-missing data
    ax2.axvline(x=promo_datetime, color='red', linestyle='--', label='Promo Date')

    # Adding dots for the start and end of each contiguous chunk
    ax2.plot(all_items[start_points].index, all_items[start_points], linewidth=0, color='#ff8500', markersize=5, marker='o', label='Start Extant Data')
    ax2.plot(all_items[end_points].index, all_items[end_points], linewidth=0, color='#ffa500', markersize=5, marker='o', label='End Extant Data')

    # Ticks and limits
    xticks = pd.date_range(promo_datetime - pd.DateOffset(days=60), periods=10, freq="15D")
    if subset:
        ax2.set_xticks(ticks=xticks)
        ax2.set_xticklabels(labels=xticks.date, rotation=70)
        ax2.set_xlim(promo_datetime - pd.DateOffset(days=60), promo_datetime + pd.DateOffset(days=60))
    ax2.set_ylim(0, max(max_ylim, all_items['item_quantity'].max()))

    # Axis and title
    ax2.set_title(f'Total Item Quantity Sold for {loc_id}')
    ax2.set_ylabel('Quantity')
    ax2.set_xlabel('Date')
    ax2.legend()

    # Save file externally
    #plt.savefig(f"Restaurant Sales Near Promo Introduction {loc_id}.png", bbox_inches='tight')

    return fig

In [ ]:
best_restaurant_ordering = data_coverage.sort_values('b2m_12H_coverage', ascending=False)['loc_id'].tolist()

In [ ]:
before_after_details.loc[best_restaurant_ordering]

Time Differences

In [ ]:
time_differences_details = {}
time_differences = {}
for loc_id in best_restaurant_ordering:

    df = sales_and_menu_data[loc_id]

    # Group by transactions (at the same time)
    transactions = df.groupby('created_at').agg({'item_quantity' : 'sum'})

    # Group by individuals days and days of the week
    transaction_by_dayofweek = transactions.groupby([transactions.index.dayofweek, transactions.index.date])

    # Take the index at every group and find the difference between time points (dropping the NaT edges) and convert to hours
    time_diffs_on_dayofweek = transaction_by_dayofweek.apply(lambda s: s.index.to_series().diff().dropna().dt.seconds//3600)

    # Store entire pivoted data
    time_differences_details[loc_id] = time_diffs_on_dayofweek

    time_diff_frequencies_list = []
    for dayofweek in range(7):

        # Subset to given day of the week and calculate the frequencies
        if dayofweek in time_diffs_on_dayofweek.index.get_level_values(0):
            time_diff_frequencies_specific_day = time_diffs_on_dayofweek[dayofweek].value_counts()
            time_diff_frequencies_specific_day.index.name = "time_diffs"
            time_diff_frequencies_specific_day.name = dayofweek
            time_diff_frequencies_list.append(pd.DataFrame(time_diff_frequencies_specific_day))

    time_diff_frequencies = time_diff_frequencies_list[0].join(time_diff_frequencies_list[1:], how='outer').sort_index()
    time_diff_frequencies = time_diff_frequencies.rename(columns={0:'Monday',1:'Tuesday',2:'Wednesday',3:'Thursday',4:'Friday',5:'Saturday',6:'Sunday'})
    time_diff_frequencies.columns.name = loc_id

    time_differences[loc_id] = time_diff_frequencies

In [ ]:
def plot_time_spacing(loc_id):

    fig, ax = plt.subplots(figsize=(10, 5))

    # Assuming time_differences[loc_id] is a valid 2D array and using pandas DataFrame
    dat = time_differences[loc_id].T.iloc[:,1:]  # Converting to DataFrame if not already one

    cax = ax.imshow(dat, cmap='viridis')  # Display the data as an image

    # Adding a color bar
    fig.colorbar(cax, ax=ax)

    # Loop over data dimensions and create text annotations for each cell
    num_rows, num_cols = dat.shape
    for i in range(num_rows):
        for j in range(num_cols):
            # Only annotate if the value is not NaN
            if dat.iloc[i, j] == dat.iloc[i, j]:
                # Rounded value if not NaN
                rounded_value = round(100 * dat.iloc[i, j]) // 100
                ax.text(j, i, str(rounded_value), ha='center', va='center', color='w', fontsize=8)

    # y_axis = dat.index.tolist()

    # # Creating a dictionary mapping column indices to days of the week
    # days_dict = {0: "Monday", 1: "Tuesday", 2: "Wednesday", 3: "Thursday", 4: "Friday", 5: "Saturday", 6: "Sunday"}

    # # Creating a dictionary mapping column indices to days of the week
    # days_dict_reverse = {"Monday":0, "Tuesday":1, "Wednesday":2, "Thursday":3, "Friday":4, "Saturday":5, "Sunday":6}

    # day_labels_reverse = [days_dict_reverse.get(day, 0) for day in y_axis]

    # # Apply the day labels based on existing columns in the data
    # day_labels = [days_dict.get(i % 7, '') for i in y_axis]


    # Assigning labels for each column with the days of the week
    days_of_week = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

    # Setting x-axis ticks to be centered on each column
    ax.set_yticks(np.arange(num_rows))

    ax.set_yticklabels(days_of_week)

    # # Ensuring the labels are displayed at the top
    # ax.xaxis.set_ticks_position('top')

    ax.set_xticks(np.arange(num_cols))
    ax.set_xticklabels(dat.columns)

    plt.show()

In [ ]:
# Turn off auto display
plt.ioff()

freq1 = 'D'
freq2 = 'W'
max_ylim1 = 0
max_ylim2 = 0
max_ylim3 = 0

for i, row in data_coverage.sort_values('b2m_12H_coverage', ascending=False).iterrows():

    # Location in order
    loc_id = row.iloc[0]

    if loc_id != 'L3XS7WSJ4AJA3':

        # Relevant DF
        df = sales_and_menu_data[loc_id]

        # Subset
        promo_datetime = pd.to_datetime(before_after_details.loc[loc_id, 'cross_over_date']).tz_localize('UTC')
        two_months_before = promo_datetime - pd.DateOffset(months=2)
        two_months_after = promo_datetime + pd.DateOffset(months=2)
        period = df.loc[two_months_before:two_months_after]

        # Calculate max y limit for plotting
        all_items1 = period.resample(freq1)['item_quantity'].sum()
        max_ylim1 = max(max_ylim1, all_items1.max())

        # Calculate max y limit for plotting
        all_items2 = period.resample(freq2)['item_quantity'].sum()
        max_ylim2 = max(max_ylim2, all_items2.max())

        # Calculate max y limit for plotting
        all_items3 = df.resample(freq2)['item_quantity'].sum()
        max_ylim3 = max(max_ylim3, all_items3.max())

        # Summary stats
        print('\n\n\n\n')
        display(Markdown(f'## {loc_id}'))
        print(locations.query('location_id == @loc_id')[['cuisine', 'city',	'state', 'restaurant_type']].to_markdown())
        print(f'\n{row.to_frame().T.to_markdown()}')
        print(f'\n{time_differences[loc_id].iloc[:2,].to_markdown()}')
        print(f'\n{sales_and_menu_data[loc_id]["item_name"].value_counts().sort_values(ascending=False).iloc[:7].to_frame().T.to_markdown()}')

        # Daily
        plot_time_spacing(loc_id)
        plot_time_series(loc_id, df, max_ylim=max_ylim1, freq=freq1)
        plot_time_series(loc_id, df, max_ylim=max_ylim2, freq=freq2)
        plot_time_series(loc_id, df, max_ylim=max_ylim3, freq=freq2, subset=False)

        # Show
        plt.show()

10 Hour Difference

In [ ]:
sales_and_menu_data[best_restaurant_ordering[0]]['item_name'].value_counts().sort_values(ascending=False).head(10)

In [ ]:
sales_and_menu_data[best_restaurant_ordering[0]].loc[pd.Timestamp('2019-07-17 4:45:58+00:00'):pd.Timestamp('2019-07-18').tz_localize('UTC')].head(5)

20 Hour Difference

In [ ]:
sales_and_menu_data[best_restaurant_ordering[0]].loc[pd.Timestamp('2020-06-07 22:27:20+00:00'):pd.Timestamp('2020-06-09').tz_localize('UTC')].head(10)

In [ ]:
time_differences['2HRX9P6HKXA8V']

In [ ]:
time_differences_details['2HRX9P6HKXA8V'][0][time_differences_details['2HRX9P6HKXA8V'][0] == 5]

In [ ]:
sales_and_menu_data['2HRX9P6HKXA8V'].loc['2020-08-24 4:55:15+00:00':'2020-08-25 10:20:15+00:00']